# 🥇 Gold — star schema (Lingua Foundry)

Builds conformed **dimensions** and **facts** from Silver for a Direct Lake
Power BI model. Grain:
- `fact_submission` — one row per submitted worksheet,
- `fact_response`   — one row per exercise response,
- `fact_exercise_score` — one row per standalone *Check*,
- `fact_conversation`   — one row per conversation (carries `news_id` → news),
- `fact_news_engagement`— conversation × news (news-grounded chats only).

Dimensions: `dim_user`, `dim_date`, `dim_language`, `dim_cefr`, `dim_scenario`, `dim_news`.
Output → `LH_LanguageApp_Gold` (overwrite). Schedule after the Silver notebook.

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
workspace        = "LanguageApp"
silver_lakehouse = "LH_LanguageApp_Silver"
gold_lakehouse   = "LH_LanguageApp_Gold"

In [ ]:
from pyspark.sql import functions as F, types as T

ONELAKE = "onelake.dfs.fabric.microsoft.com"

def tables_base(lakehouse: str) -> str:
    lh = lakehouse if (lakehouse.endswith(".Lakehouse") or "-" in lakehouse and len(lakehouse) == 36) else f"{lakehouse}.Lakehouse"
    return f"abfss://{workspace}@{ONELAKE}/{lh}/Tables"

def read_delta(lakehouse: str, table: str, schema: str | None = None):
    base = tables_base(lakehouse)
    path = f"{base}/{schema}/{table}" if schema else f"{base}/{table}"
    return spark.read.format("delta").load(path)

def write_delta(df, lakehouse: str, table: str, schema: str | None = None):
    base = tables_base(lakehouse)
    path = f"{base}/{schema}/{table}" if schema else f"{base}/{table}"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").save(path))
    print(f"  wrote {df.count():>7} -> {lakehouse}/{schema + '/' if schema else ''}{table}")

## Read Silver

In [ ]:
s_users   = read_delta(silver_lakehouse, "users")
s_lessons = read_delta(silver_lakehouse, "lessons")
s_subs    = read_delta(silver_lakehouse, "worksheet_submissions")
s_resp    = read_delta(silver_lakehouse, "worksheet_responses")
s_scores  = read_delta(silver_lakehouse, "exercise_scores")
s_convos  = read_delta(silver_lakehouse, "conversations")
s_date    = read_delta(silver_lakehouse, "date_dim")
s_news    = read_delta(silver_lakehouse, "news")

## Dimensions

In [ ]:
# dim_user
dim_user = s_users.select(
    F.col("id").alias("user_id"), "user_name", "native_language",
    "created_date_key", "is_sample")

# dim_date (straight from the conformed calendar)
dim_date = s_date

# dim_language — codes used across learner + news data
langs = (s_lessons.select(F.col("target_language").alias("language_code"))
         .union(s_news.select(F.col("language").alias("language_code")))
         .union(s_convos.select(F.col("target_language").alias("language_code")))
         .distinct().filter(F.col("language_code").isNotNull()))
name_map = F.create_map(F.lit("en"), F.lit("English"),
                        F.lit("es"), F.lit("Spanish"),
                        F.lit("fr"), F.lit("French"))
dim_language = langs.withColumn("language_name",
        F.coalesce(name_map[F.col("language_code")], F.initcap("language_code")))

# dim_cefr — ordered CEFR levels
cefr_rows = [("A1",1),("A2",2),("B1",3),("B2",4),("C1",5),("C2",6)]
dim_cefr = spark.createDataFrame(cefr_rows, ["cefr_level","cefr_order"])

# dim_scenario — scenario/verb practice contexts from submissions
dim_scenario = (s_subs.select(
        F.coalesce(F.col("scenario"), F.lit("(none)")).alias("scenario_name"),
        F.col("mode")).distinct()
    .withColumn("scenario_key", F.sha2(F.concat_ws("|", "scenario_name", "mode"), 256)))

# dim_news — one row per enriched news item
dim_news = s_news.select(
    "news_id", "language", "cefr_level", "domain", "source_country",
    "title_translated", "summary", "english_gloss", "seen_date_key")

## Facts

In [ ]:
# fact_submission (grain: submission)
fact_submission = s_subs.select(
    "submission_id", "lesson_id",
    F.col("user_id"), F.col("date_key"),
    F.col("target_language").alias("language_code"),
    F.col("difficulty").alias("cefr_level"),
    F.coalesce(F.col("scenario"), F.lit("(none)")).alias("scenario_name"),
    "mode", "verb", "grammar_focus",
    "total_exercises", "answered_count",
    "first_correct_count", "final_correct_count",
    "first_score_avg", "final_score_avg",
    "improvement", "accuracy_first", "accuracy_final", "submitted_ts")

# fact_response (grain: exercise response)
fact_response = s_resp.select(
    "response_id", "submission_id", "user_id", "exercise_id", "date_key",
    F.col("target_language").alias("language_code"),
    F.col("difficulty").alias("cefr_level"),
    "exercise_type", "first_score", "first_is_correct",
    "final_score", "final_is_correct", "attempts")

# fact_exercise_score (grain: standalone Check)
fact_exercise_score = s_scores.select(
    "id", "exercise_id", "user_id", "date_key",
    "score", "is_correct")

# fact_conversation (grain: conversation) — carries news_id for the news join
fact_conversation = s_convos.select(
    "id", "user_id", "date_key",
    F.col("target_language").alias("language_code"),
    "news_id", "is_news_grounded",
    "turn_count", "user_turns", "corrected_turns", "duration_min")

# fact_news_engagement (grain: news-grounded conversation × news)
fact_news_engagement = (s_convos.filter(F.col("news_id").isNotNull())
    .join(s_news.select("news_id", F.col("language").alias("news_language"),
                        F.col("cefr_level").alias("news_cefr"), "domain"),
          "news_id", "left")
    .select("id", "user_id", "date_key", "news_id",
            "news_language", "news_cefr", "domain",
            "turn_count", "duration_min"))

## Write Gold

In [ ]:
print("Writing Gold dimensions…")
for name, df in {
    "dim_user": dim_user, "dim_date": dim_date, "dim_language": dim_language,
    "dim_cefr": dim_cefr, "dim_scenario": dim_scenario, "dim_news": dim_news,
}.items():
    write_delta(df, gold_lakehouse, name)

print("Writing Gold facts…")
for name, df in {
    "fact_submission": fact_submission, "fact_response": fact_response,
    "fact_exercise_score": fact_exercise_score, "fact_conversation": fact_conversation,
    "fact_news_engagement": fact_news_engagement,
}.items():
    write_delta(df, gold_lakehouse, name)
print("Gold star schema complete.")

## 🔗 Power BI model (Direct Lake)

Build a Direct Lake semantic model on `LH_LanguageApp_Gold` with these relationships
(single-direction, one→many from dim to fact):

| From (dim) | key | To (fact) |
|---|---|---|
| `dim_user[user_id]` | → | `fact_*[user_id]` |
| `dim_date[date_key]` | → | `fact_*[date_key]` |
| `dim_language[language_code]` | → | `fact_submission / fact_response / fact_conversation[language_code]` |
| `dim_cefr[cefr_level]` | → | `fact_submission / fact_response[cefr_level]` |
| `dim_scenario[scenario_name+mode]` | → | `fact_submission[scenario_name+mode]` |
| `dim_news[news_id]` | → | `fact_conversation / fact_news_engagement[news_id]` |

Mark `dim_date` as the date table. Example measures:
- **Avg improvement** = `AVERAGE(fact_submission[improvement])`
- **First-try accuracy** = `AVERAGE(fact_submission[accuracy_first])`
- **News-grounded chats %** = `DIVIDE(CALCULATE(COUNTROWS(fact_conversation), fact_conversation[is_news_grounded]), COUNTROWS(fact_conversation))`